# ScavenPred — frozen model reproduction

Reproducibility notebook for the frozen revision analysis. It trains the Random Forest only from the packaged training data, computes SHAP values, and exports the fitted model. Dataset-audit and benchmark outputs used in the revision are supplied separately in the package.

In [ ]:
import pandas as pd, numpy as np, joblib, shap
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path

In [ ]:
FEATURES = ['E5', 'ProtFP3', 'BLOSUM6', 'BLOSUM3', 'Z3']
MODEL_PARAMS = {'n_estimators': 1000, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1}
FEATURES, MODEL_PARAMS

In [ ]:
ROOT = Path.cwd().parent
path = ROOT / "data" / 'ScavenPred_training_1648_with_88_features.xlsx'
xls = pd.ExcelFile(path)
print(xls.sheet_names)


In [ ]:
df = pd.read_excel(path, sheet_name="Training_88_Features")
X = df[FEATURES].copy()
y = df["class"].astype(int)
print(df.shape, y.value_counts().to_dict())

In [ ]:
rf = RandomForestClassifier(**MODEL_PARAMS)
rf.fit(X, y)
joblib.dump(rf, ROOT/"results"/"frozen_random_forest.joblib")
print("Saved fitted model.")

In [ ]:
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X)
arr = np.asarray(sv)
if isinstance(sv, list):
    sv_pos = np.asarray(sv[1])
elif arr.ndim == 3:
    sv_pos = arr[:, :, 1]
else:
    sv_pos = arr
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Mean_abs_SHAP": np.abs(sv_pos).mean(axis=0)
}).sort_values("Mean_abs_SHAP", ascending=False)
importance

In [ ]:
importance.to_excel(ROOT/"results"/"reproduced_SHAP_global_importance.xlsx", index=False)
print("Done.")